In [0]:
%sql
WITH source_customers AS (
    SELECT
        CAST(customer_id AS STRING) AS customer_code,
        customer,
        market,
        platform,
        channel
    FROM fmcg.gold.sb_dim_customers
),
current_target AS (
    SELECT
        customer_code,
        customer,
        market,
        platform,
        channel
    FROM fmcg.gold.dim_customers_scd2_practice
    WHERE is_current = true
),
classified AS (
    SELECT
        COALESCE(s.customer_code, t.customer_code) AS customer_code,
        s.customer AS source_customer,
        t.customer AS target_customer,
        s.market AS source_market,
        t.market AS target_market,
        CASE
            WHEN s.customer_code IS NULL THEN 'TARGET_ONLY'
            WHEN t.customer_code IS NULL THEN 'NEW'
            WHEN NOT (
                s.customer <=> t.customer
                AND s.market <=> t.market
                AND s.platform <=> t.platform
                AND s.channel <=> t.channel
            ) THEN 'CHANGED'
            ELSE 'UNCHANGED'
        END AS change_type
    FROM source_customers s
    FULL OUTER JOIN current_target t
        ON s.customer_code = t.customer_code
)
SELECT
    change_type,
    COUNT(*) AS customer_count
FROM classified
GROUP BY change_type
ORDER BY change_type;

In [0]:
from pyspark.sql import functions as F

source_table = "fmcg.gold.sb_dim_customers"
target_table = "fmcg.gold.dim_customers_scd2_practice"

source_df = (
    spark.table(source_table)
    .select(
        F.col("customer_id").cast("string").alias("customer_code"),
        "customer",
        "market",
        "platform",
        "channel"
    )
)

# Check whether source customer codes are duplicated
duplicate_source_codes = (
    source_df
    .groupBy("customer_code")
    .count()
    .filter(
        F.col("customer_code").isNull() |
        (F.col("count") > 1)
    )
)

display(duplicate_source_codes)

In [0]:
from pyspark.sql import functions as F

target_current_df = (
    spark.table(target_table)
    .filter(F.col("is_current") == True)
    .select(
        "customer_code",
        "customer",
        "market",
        "platform",
        "channel"
    )
)

s = source_df.alias("s")
t = target_current_df.alias("t")

comparison_df = (
    s.join(
        t,
        F.col("s.customer_code") == F.col("t.customer_code"),
        "full_outer"
    )
    .select(
        F.coalesce(
            F.col("s.customer_code"),
            F.col("t.customer_code")
        ).alias("customer_code"),

        F.col("s.customer").alias("source_customer"),
        F.col("t.customer").alias("target_customer"),

        F.col("s.market").alias("source_market"),
        F.col("t.market").alias("target_market"),

        F.col("s.platform").alias("source_platform"),
        F.col("t.platform").alias("target_platform"),

        F.col("s.channel").alias("source_channel"),
        F.col("t.channel").alias("target_channel"),

        F.when(
            F.col("s.customer_code").isNull(),
            "TARGET_ONLY"
        )
        .when(
            F.col("t.customer_code").isNull(),
            "NEW"
        )
        .when(
            ~(
                F.col("s.customer").eqNullSafe(F.col("t.customer"))
                & F.col("s.market").eqNullSafe(F.col("t.market"))
                & F.col("s.platform").eqNullSafe(F.col("t.platform"))
                & F.col("s.channel").eqNullSafe(F.col("t.channel"))
            ),
            "CHANGED"
        )
        .otherwise("UNCHANGED")
        .alias("change_type")
    )
)

display(
    comparison_df
    .groupBy("change_type")
    .count()
    .orderBy("change_type")
)

In [0]:
# Rows whose existing current version must be expired
rows_to_expire = (
    comparison_df
    .filter(F.col("change_type") == "CHANGED")
    .select("customer_code")
)

# New current versions for changed customers
changed_rows_to_insert = (
    comparison_df
    .filter(F.col("change_type") == "CHANGED")
    .select(
        "customer_code",
        F.col("source_customer").alias("customer"),
        F.col("source_market").alias("market"),
        F.col("source_platform").alias("platform"),
        F.col("source_channel").alias("channel")
    )
)

# First versions for genuinely new customers
new_rows_to_insert = (
    comparison_df
    .filter(F.col("change_type") == "NEW")
    .select(
        "customer_code",
        F.col("source_customer").alias("customer"),
        F.col("source_market").alias("market"),
        F.col("source_platform").alias("platform"),
        F.col("source_channel").alias("channel")
    )
)

print("Current versions to expire:", rows_to_expire.count())
print("Changed versions to insert:", changed_rows_to_insert.count())
print("New customer versions to insert:", new_rows_to_insert.count())

display(changed_rows_to_insert)

In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F

target_delta = DeltaTable.forName(spark, target_table)

current_counts = (
    spark.table(target_table)
    .filter(
        F.col("customer_code").isin(
            [row["customer_code"] for row in rows_to_expire.collect()]
        )
        & (F.col("is_current") == True)
    )
    .groupBy("customer_code")
    .count()
)

invalid_expirations = current_counts.filter(F.col("count") != 1)

if invalid_expirations.count() > 0:
    raise ValueError(
        "Safety check failed: a changed customer does not have exactly one current version."
    )

print("Safety check passed. Ready to expire changed versions.")

In [0]:

from delta.tables import DeltaTable
from pyspark.sql import functions as F


def run_scd2():
    # --------------------------------------------------
    # 1. Configuration
    # --------------------------------------------------
    source_table = "fmcg.gold.sb_dim_customers"
    target_table = "fmcg.gold.dim_customers_scd2_practice"

    tracked_columns = ["customer", "market", "platform", "channel"]

    # Use one timestamp for this entire run
    run_timestamp = spark.sql(
        "SELECT current_timestamp() AS ts"
    ).first()["ts"]

    # --------------------------------------------------
    # 2. Read fresh source data
    # --------------------------------------------------
    source = (
        spark.table(source_table)
        .select(
            F.col("customer_id").cast("string").alias("customer_code"),
            *tracked_columns
        )
    )

    # Validate source keys
    if source.filter(
        F.col("customer_code").isNull()
    ).limit(1).count() > 0:
        raise ValueError(
            "SCD2 stopped: source contains null customer codes."
        )

    duplicate_source_keys = (
        source.groupBy("customer_code")
        .count()
        .filter(F.col("count") > 1)
    )

    if duplicate_source_keys.limit(1).count() > 0:
        raise ValueError(
            "SCD2 stopped: source contains duplicate customer codes."
        )

    source_keys = source.select("customer_code").distinct()

    # --------------------------------------------------
    # 3. Validate target before making changes
    # --------------------------------------------------
    target = spark.table(target_table)

    if target.filter(
        F.col("customer_code").isNull()
    ).limit(1).count() > 0:
        raise ValueError(
            "SCD2 stopped: target contains null customer codes."
        )

    if target.filter(
        F.col("is_current").isNull()
    ).limit(1).count() > 0:
        raise ValueError(
            "SCD2 stopped: target contains null is_current values."
        )

    if target.filter(
        F.col("effective_start_date").isNull()
    ).limit(1).count() > 0:
        raise ValueError(
            "SCD2 stopped: target contains null start dates."
        )

    if target.filter(
        (F.col("is_current") == True)
        & F.col("effective_end_date").isNotNull()
    ).limit(1).count() > 0:
        raise ValueError(
            "SCD2 stopped: a current version has an end date."
        )

    if target.filter(
        (F.col("is_current") == False)
        & F.col("effective_end_date").isNull()
    ).limit(1).count() > 0:
        raise ValueError(
            "SCD2 stopped: a historical version has no end date."
        )

    # Count current versions for each existing customer
    target_current_counts = (
        target.groupBy("customer_code")
        .agg(
            F.sum(
                F.when(F.col("is_current") == True, 1)
                 .otherwise(0)
            ).alias("current_versions")
        )
    )

    # Multiple current versions are always unsafe
    multiple_current_versions = target_current_counts.filter(
        F.col("current_versions") > 1
    )

    if multiple_current_versions.limit(1).count() > 0:
        raise ValueError(
            "SCD2 stopped: a customer has multiple current versions."
        )

    # Recovery case:
    # A previous run may have expired a current row but failed
    # before inserting its replacement.
    zero_current_versions = target_current_counts.filter(
        F.col("current_versions") == 0
    )

    # A zero-current customer can be recovered only if it exists
    # in the source. Target-only customers must not be changed
    # automatically because source absence does not mean deletion.
    unrecoverable_keys = zero_current_versions.join(
        source_keys,
        on="customer_code",
        how="left_anti"
    )

    if unrecoverable_keys.limit(1).count() > 0:
        raise ValueError(
            "SCD2 stopped: a target-only customer has no current "
            "version. Investigate this customer before proceeding."
        )

    # --------------------------------------------------
    # 4. Expire changed current versions
    # --------------------------------------------------
    change_condition = " OR ".join(
        [
            f"NOT (t.`{column}` <=> s.`{column}`)"
            for column in tracked_columns
        ]
    )

    delta_target = DeltaTable.forName(spark, target_table)

    (
        delta_target.alias("t")
        .merge(
            source.alias("s"),
            """
            t.customer_code = s.customer_code
            AND t.is_current = true
            """
        )
        .whenMatchedUpdate(
            condition=change_condition,
            set={
                "effective_end_date": F.lit(run_timestamp),
                "is_current": F.lit(False)
            }
        )
        .execute()
    )

    # --------------------------------------------------
    # 5. Re-read target and find customers needing a
    #    current version (new or recovery)
    # --------------------------------------------------
    current_keys = (
        spark.table(target_table)
        .filter(F.col("is_current") == True)
        .select("customer_code")
        .distinct()
    )

    rows_to_insert = source.join(
        current_keys,
        on="customer_code",
        how="left_anti"
    )

    # Build replacement/new current versions
    insert_rows = rows_to_insert.select(
        "customer_code",
        "customer",
        "market",
        "platform",
        "channel",
        F.lit(run_timestamp).cast("timestamp")
         .alias("effective_start_date"),
        F.lit(None).cast("timestamp")
         .alias("effective_end_date"),
        F.lit(True).alias("is_current")
    )

    insert_count = insert_rows.count()

    # Delta append is atomic as an individual write.
    # It is separate from the expiration MERGE above.
    if insert_count > 0:
        (
            insert_rows.write
            .format("delta")
            .mode("append")
            .saveAsTable(target_table)
        )

    # --------------------------------------------------
    # 6. Validate the result
    # --------------------------------------------------
    final_target = spark.table(target_table)

    final_current_counts = (
        final_target.groupBy("customer_code")
        .agg(
            F.sum(
                F.when(F.col("is_current") == True, 1)
                 .otherwise(0)
            ).alias("current_versions")
        )
    )

    invalid_final_counts = final_current_counts.filter(
        F.col("current_versions") != 1
    )

    if invalid_final_counts.limit(1).count() > 0:
        raise ValueError(
            "SCD2 validation failed: at least one customer does not "
            "have exactly one current version. Investigate before rerunning."
        )

    invalid_final_dates = final_target.filter(
        (
            (F.col("is_current") == True)
            & F.col("effective_end_date").isNotNull()
        )
        |
        (
            (F.col("is_current") == False)
            & F.col("effective_end_date").isNull()
        )
    )

    if invalid_final_dates.limit(1).count() > 0:
        raise ValueError(
            "SCD2 validation failed: current/historical dates are inconsistent."
        )

    # --------------------------------------------------
    # 7. Report results
    # --------------------------------------------------
    print("SCD2 run completed successfully.")
    print(f"Source customers: {source_keys.count()}")
    print(f"New current versions inserted: {insert_count}")
    print(f"Target table: {target_table}")


# Execute against the practice table only
run_scd2()


In [0]:
%sql
SELECT
    customer_code,
    market,
    effective_start_date,
    effective_end_date,
    is_current
FROM fmcg.gold.dim_customers_scd2_practice
WHERE customer_code = '789101'
ORDER BY effective_start_date;

In [0]:
run_scd2()

In [0]:
%sql
SELECT
    COUNT(*) AS total_versions,
    SUM(CASE WHEN is_current = true THEN 1 ELSE 0 END)
        AS current_versions,
    SUM(CASE WHEN is_current = false THEN 1 ELSE 0 END)
        AS historical_versions
FROM fmcg.gold.dim_customers_scd2_practice
WHERE customer_code = '789101';

In [0]:
%sql
SELECT
    t.customer_code,
    COUNT(*) AS total_versions,
    SUM(CASE WHEN t.is_current = true THEN 1 ELSE 0 END)
        AS current_versions,
    MAX(s.customer_id IS NOT NULL) AS exists_in_source
FROM fmcg.gold.dim_customers_scd2_practice t
LEFT JOIN fmcg.gold.sb_dim_customers s
    ON t.customer_code = CAST(s.customer_id AS STRING)
GROUP BY t.customer_code
HAVING SUM(CASE WHEN t.is_current = true THEN 1 ELSE 0 END) = 0;

In [0]:
target = spark.table(target_table)

In [0]:
# Check for customers with zero current versions
invalid_current_counts = (
    target.groupBy("customer_code")
    .agg(
        F.sum(
            F.when(F.col("is_current") == True, 1)
             .otherwise(0)
        ).alias("current_versions")
    )
    .filter(F.col("current_versions") != 1)
)

if invalid_current_counts.limit(1).count() > 0:
    raise ValueError(
        "SCD2 stopped: some customers do not have exactly "
        "one current version. Investigate before processing."
    )

In [0]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT customer_code) AS unique_customers,
    SUM(CASE WHEN is_current = true THEN 1 ELSE 0 END)
        AS current_rows,
    SUM(CASE WHEN is_current = false THEN 1 ELSE 0 END)
        AS historical_rows
FROM fmcg.gold.dim_customers_scd2_practice;

In [0]:
%sql
WITH customer_history AS (
    SELECT
        customer_code,
        market,
        effective_start_date,
        effective_end_date,
        is_current,
        LEAD(effective_start_date) OVER (
            PARTITION BY customer_code
            ORDER BY effective_start_date
        ) AS next_start_date
    FROM fmcg.gold.dim_customers_scd2_practice
)
SELECT *
FROM customer_history
WHERE
    (
        is_current = false
        AND (
            effective_end_date IS NULL
            OR next_start_date IS NULL
            OR effective_end_date <> next_start_date
        )
    )
    OR (
        is_current = true
        AND effective_end_date IS NOT NULL
    );

In [0]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT customer_code) AS unique_customers,
    SUM(CASE WHEN is_current = true THEN 1 ELSE 0 END)
        AS current_rows,
    SUM(CASE WHEN is_current = false THEN 1 ELSE 0 END)
        AS historical_rows
FROM fmcg.gold.dim_customers_scd2_practice;

In [0]:
%sql
SELECT
    customer_code,
    customer,
    market,
    platform,
    channel,
    effective_start_date,
    effective_end_date,
    is_current
FROM fmcg.gold.dim_customers_scd2_practice
WHERE customer_code = '789101'
ORDER BY effective_start_date;


In [0]:
%sql
SELECT
    customer_code,
    customer,
    market,
    platform,
    channel,
    effective_start_date,
    effective_end_date,
    is_current
FROM fmcg.gold.dim_customers_scd2_practice
WHERE customer_code IN (
    SELECT customer_code
    FROM fmcg.gold.dim_customers_scd2_practice
    GROUP BY customer_code
    HAVING COUNT(*) = 1
       AND SUM(CASE WHEN is_current = true THEN 1 ELSE 0 END) = 1
)
ORDER BY customer_code
LIMIT 10;

In [0]:
%sql
CREATE OR REPLACE TABLE fmcg.gold.sb_dim_customers_scd2_test
AS
SELECT *
FROM fmcg.gold.sb_dim_customers;

In [0]:
%sql
SELECT customer_id, customer_name, city, customer, market, platform, channel
FROM fmcg.gold.sb_dim_customers_scd2_test
WHERE CAST(customer_id AS STRING) = '789102';

In [0]:
%sql
UPDATE fmcg.gold.sb_dim_customers_scd2_test
SET market = 'USA'
WHERE CAST(customer_id AS STRING) = '789102';

In [0]:
%sql
SELECT
    customer_id,
    customer_name,
    market
FROM fmcg.gold.sb_dim_customers_scd2_test
WHERE CAST(customer_id AS STRING) = '789102';

In [0]:

from delta.tables import DeltaTable
from pyspark.sql import functions as F

def run_scd2_test():
    source_table = "fmcg.gold.sb_dim_customers_scd2_test"
    target_table = "fmcg.gold.dim_customers_scd2_practice"
    tracked_columns = ["customer", "market", "platform", "channel"]

    run_timestamp = spark.sql(
        "SELECT current_timestamp() AS ts"
    ).first()["ts"]

    # Read test source
    source = (
        spark.table(source_table)
        .select(
            F.col("customer_id").cast("string").alias("customer_code"),
            *tracked_columns
        )
    )

    # Validate source keys
    if source.filter(F.col("customer_code").isNull()).limit(1).count():
        raise ValueError("Test stopped: source has null customer codes.")

    duplicates = (
        source.groupBy("customer_code")
        .count()
        .filter(F.col("count") > 1)
    )
    if duplicates.limit(1).count():
        raise ValueError("Test stopped: source has duplicate customer codes.")

    target = spark.table(target_table)

    # Refuse to proceed if the target already has multiple current versions
    multiple_current = (
        target.filter(F.col("is_current") == True)
        .groupBy("customer_code")
        .count()
        .filter(F.col("count") > 1)
    )
    if multiple_current.limit(1).count():
        raise ValueError("Test stopped: target has multiple current versions.")

    # A missing current version is recoverable only if the customer
    # exists in the test source. Don't silently repair target-only keys.
    current_counts = (
        target.groupBy("customer_code")
        .agg(
            F.sum(
                F.when(F.col("is_current") == True, 1).otherwise(0)
            ).alias("current_versions")
        )
    )

    zero_current = current_counts.filter(F.col("current_versions") == 0)
    source_keys = source.select("customer_code").distinct()

    target_only_zero_current = zero_current.join(
        source_keys, on="customer_code", how="left_anti"
    )
    if target_only_zero_current.limit(1).count():
        raise ValueError(
            "Test stopped: a target-only customer has no current version."
        )

    # Expire changed current versions
    change_condition = " OR ".join(
        [f"NOT (t.`{c}` <=> s.`{c}`)" for c in tracked_columns]
    )

    (
        DeltaTable.forName(spark, target_table)
        .alias("t")
        .merge(
            source.alias("s"),
            """
            t.customer_code = s.customer_code
            AND t.is_current = true
            """
        )
        .whenMatchedUpdate(
            condition=change_condition,
            set={
                "effective_end_date": F.lit(run_timestamp),
                "is_current": F.lit(False)
            }
        )
        .execute()
    )

    # Insert a current version for each source customer that has none
    current_keys = (
        spark.table(target_table)
        .filter(F.col("is_current") == True)
        .select("customer_code")
        .distinct()
    )

    rows_to_insert = source.join(
        current_keys, on="customer_code", how="left_anti"
    )

    insert_rows = rows_to_insert.select(
        "customer_code",
        "customer",
        "market",
        "platform",
        "channel",
        F.lit(run_timestamp).cast("timestamp").alias("effective_start_date"),
        F.lit(None).cast("timestamp").alias("effective_end_date"),
        F.lit(True).alias("is_current")
    )

    insert_count = insert_rows.count()

    if insert_count:
        (
            insert_rows.write
            .format("delta")
            .mode("append")
            .saveAsTable(target_table)
        )

    # Final invariant: every target customer must have exactly one current version
    final_counts = (
        spark.table(target_table)
        .groupBy("customer_code")
        .agg(
            F.sum(
                F.when(F.col("is_current") == True, 1).otherwise(0)
            ).alias("current_versions")
        )
        .filter(F.col("current_versions") != 1)
    )

    if final_counts.limit(1).count():
        raise ValueError(
            "Test validation failed: a customer does not have exactly "
            "one current version. Investigate before rerunning."
        )

    print("Test SCD2 run completed.")
    print(f"New current versions inserted: {insert_count}")
    print(f"Target: {target_table}")

run_scd2_test()


In [0]:
%sql
SELECT
    customer_code,
    customer,
    market,
    platform,
    channel,
    effective_start_date,
    effective_end_date,
    is_current
FROM fmcg.gold.dim_customers_scd2_practice
WHERE customer_code = '789102'
ORDER BY effective_start_date;

In [0]:
%sql
SELECT
    COUNT(*) AS total_versions,
    SUM(CASE WHEN is_current = true THEN 1 ELSE 0 END) AS current_versions,
    SUM(CASE WHEN is_current = false THEN 1 ELSE 0 END) AS historical_versions
FROM fmcg.gold.dim_customers_scd2_practice
WHERE customer_code = '789102';

In [0]:
%sql
UPDATE fmcg.gold.sb_dim_customers_scd2_test
SET market = 'India'
WHERE CAST(customer_id AS STRING) = '789102';


In [0]:
%sql
SELECT
    customer_id,
    customer_name,
    market
FROM fmcg.gold.sb_dim_customers_scd2_test
WHERE CAST(customer_id AS STRING) = '789102';

In [0]:
%sql
SELECT
    customer_code,
    market,
    effective_start_date,
    effective_end_date,
    is_current
FROM fmcg.gold.dim_customers_scd2_practice
WHERE customer_code = '789102'
ORDER BY effective_start_date;

In [0]:
%sql
CREATE OR REPLACE TABLE fmcg.gold.dim_customers_scd2_recovery_test
AS
SELECT *
FROM fmcg.gold.dim_customers_scd2_practice
WHERE customer_code = '789102';

In [0]:
%sql
SELECT
    customer_code,
    market,
    effective_start_date,
    effective_end_date,
    is_current
FROM fmcg.gold.dim_customers_scd2_recovery_test
ORDER BY effective_start_date;

In [0]:
%sql
UPDATE fmcg.gold.dim_customers_scd2_recovery_test
SET
    effective_end_date = current_timestamp(),
    is_current = false
WHERE customer_code = '789102'
  AND is_current = true;

In [0]:
%sql
SELECT
    customer_code,
    market,
    effective_start_date,
    effective_end_date,
    is_current
FROM fmcg.gold.dim_customers_scd2_recovery_test
ORDER BY effective_start_date;

In [0]:

from pyspark.sql import functions as F

source_table = "fmcg.gold.sb_dim_customers_scd2_test"
target_table = "fmcg.gold.dim_customers_scd2_recovery_test"

run_timestamp = spark.sql(
    "SELECT current_timestamp() AS ts"
).first()["ts"]

# Read the source record for the customer being recovered
source_row = (
    spark.table(source_table)
    .filter(F.col("customer_id").cast("string") == "789102")
    .select(
        F.col("customer_id").cast("string").alias("customer_code"),
        "customer",
        "market",
        "platform",
        "channel"
    )
)

if source_row.count() != 1:
    raise ValueError(
        "Recovery stopped: expected exactly one source record for 789102."
    )

# Confirm that the recovery target currently has zero current versions
target = spark.table(target_table)

current_count = (
    target.filter(
        (F.col("customer_code") == "789102")
        & (F.col("is_current") == True)
    ).count()
)

if current_count != 0:
    raise ValueError(
        "Recovery stopped: expected zero current versions before recovery."
    )

# Insert the missing current version
recovery_row = source_row.select(
    "customer_code",
    "customer",
    "market",
    "platform",
    "channel",
    F.lit(run_timestamp).cast("timestamp").alias("effective_start_date"),
    F.lit(None).cast("timestamp").alias("effective_end_date"),
    F.lit(True).alias("is_current")
)

recovery_row.write.format("delta").mode("append").saveAsTable(target_table)

print("Recovery insertion completed.")
print("Customer: 789102")
print("Market: India")
print("New current version inserted: 1")


In [0]:
%sql
SELECT
    customer_code,
    market,
    effective_start_date,
    effective_end_date,
    is_current
FROM fmcg.gold.dim_customers_scd2_recovery_test
WHERE customer_code = '789102'
ORDER BY effective_start_date;

In [0]:
%sql
SELECT
    COUNT(*) AS total_versions,
    SUM(CASE WHEN is_current = true THEN 1 ELSE 0 END)
        AS current_versions,
    SUM(CASE WHEN is_current = false THEN 1 ELSE 0 END)
        AS historical_versions,
    SUM(
        CASE
            WHEN is_current = true
             AND effective_end_date IS NOT NULL
            THEN 1
            ELSE 0
        END
    ) AS invalid_current_rows
FROM fmcg.gold.dim_customers_scd2_recovery_test
WHERE customer_code = '789102';